# RAPTOR — 계층적 요약으로 긴 문서를 정복하는 RAG

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **RAPTOR**의 계층적 요약 구조와 왜 긴 문서에 효과적인지 이해하기
- 원본 청크(레벨 0)와 그룹 요약(레벨 1)을 하나의 벡터스토어에 통합하기
- 질문의 추상화 수준에 따라 적절한 레벨에서 검색이 이루어지는 과정 확인하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인
- LLM Chain으로 문서 요약하는 방법

---

```mermaid
flowchart TD
    D[원본 문서]:::input --> L0[레벨 0<br/>원본 청크 800자]:::process
    L0 --> G[5개씩 그룹화]:::process
    G --> L1[레벨 1<br/>그룹 요약]:::process
    L0 --> VS[(통합 벡터스토어<br/>레벨 0 + 레벨 1)]:::storage
    L1 --> VS
    Q[사용자 질문]:::input --> VS
    VS --> R[레벨별 검색<br/>k=5]:::process
    R --> A[계층적 맥락<br/>답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 왜 RAPTOR가 필요한가?

| 질문 유형 | 일반 RAG | RAPTOR |
|---------|---------|--------|
| "이 논문의 주요 기여는?" | 청크 단위로 파악 어려움 | 레벨 1 요약에서 전체 파악 |
| "3장의 실험 설정은?" | 원본 청크에서 정확히 검색 | 레벨 0에서 세부 검색 |

> **실무 팁**: 이 노트북은 RAPTOR의 핵심 아이디어를 2레벨로 간소화한 버전이에요. 실제 RAPTOR 논문은 가우시안 혼합 모델 클러스터링을 활용한 3~4 레벨 구조를 사용합니다.

## 왜 RAPTOR가 필요한가?

### 일반 RAG의 한계

긴 문서(예: 100페이지 논문)에서:
- "이 논문의 주요 기여는?" → 여러 청크를 종합해야 답변 가능
- 하지만 청크는 독립적으로 검색됨
- 전체적인 맥락 파악 어려움

### RAPTOR의 해결책

- 여러 레벨의 요약 생성
- 질문에 따라 적절한 추상화 레벨 선택
- 전체 맥락과 세부 정보 모두 활용

## 학습 목표

이 노트북에서는 **간소화된 RAPTOR**를 구현합니다:
- 2레벨 계층 구조 (원본 + 요약)
- 통합 검색 (모든 레벨에서 동시 검색)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

## 1. 긴 문서 로드

In [ ]:
# 문서 로드
loader = PyMuPDFLoader("../6-1_DocumentLoaders/data/sample-rag-brief.pdf")
docs = loader.load()

# 원본 청크로 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
level0_chunks = text_splitter.split_documents(docs)

print(f"레벨 0 (원본): {len(level0_chunks)}개 청크")
print(f"평균 청크 크기: {sum(len(c.page_content) for c in level0_chunks) / len(level0_chunks):.0f} 문자")

## 2. 레벨 1 요약 생성

여러 청크를 그룹화하여 요약합니다.

In [ ]:
# ---------------------------------------------------
# 청크 그룹화 후 LLM으로 레벨 1 요약 생성
# ---------------------------------------------------
# ============================================================
# TODO: 5개 청크씩 묶어 요약 체인으로 레벨 1 요약을 만드세요
# 힌트: for i in range(0, len(level0_chunks), 5) → summary_chain.invoke(combined_text)
# 예상 결과: 각 그룹의 요약 완료 메시지 출력
# ============================================================


## 3. 계층적 벡터스토어 구축

모든 레벨의 문서를 하나의 벡터스토어에 저장합니다.

In [ ]:
# ---------------------------------------------------
# 레벨 0 + 레벨 1을 하나의 FAISS 벡터스토어로 통합
# ---------------------------------------------------
# ============================================================
# TODO: level0_chunks와 level1_summaries를 합쳐 FAISS 벡터스토어를 만드세요
# 힌트: all_documents = level0_chunks + level1_summaries, FAISS.from_documents(...)
# 예상 결과: 계층적 벡터스토어 구축 완료 (레벨별 문서 수 출력)
# ============================================================


## 4. RAPTOR RAG 체인

In [ ]:
# ---------------------------------------------------
# RAPTOR RAG 체인 구성 — 계층적 검색 + LLM 답변
# ---------------------------------------------------
# ============================================================
# TODO: retriever와 raptor_prompt를 LCEL로 연결해 RAG 체인을 만드세요
# 힌트: search_kwargs={"k": 5}로 다양한 레벨에서 검색
# 예상 결과: "RAPTOR RAG 체인 생성 완료" 출력
# ============================================================


## 5. 테스트: 추상화 레벨별 질문

In [ ]:
# ---------------------------------------------------
# 테스트: 개괄적 질문 (레벨 1 요약에서 답변 기대)
# ---------------------------------------------------
# ============================================================
# TODO: raptor_chain.invoke()로 전체 주제를 묻는 질문을 하세요
# 힌트: retriever.invoke(question)으로 검색된 문서의 레벨도 확인
# 예상 결과: 레벨 1 요약 문서가 검색되고 전체 주제에 대한 답변 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# 테스트: 세부적 질문 (레벨 0 원본에서 답변 기대)
# ---------------------------------------------------
# ============================================================
# TODO: raptor_chain.invoke()로 구체적인 추진 방법을 묻는 질문을 하세요
# 힌트: retriever.invoke(question)으로 검색된 문서의 레벨도 확인
# 예상 결과: 레벨 0 원본 문서가 검색되고 세부 내용 답변 출력
# ============================================================


## 💡 핵심 정리

### RAPTOR의 핵심 아이디어

1. **계층적 요약**: 문서를 여러 추상화 레벨로 표현
2. **통합 검색**: 모든 레벨에서 동시 검색
3. **유연한 답변**: 질문의 특성에 따라 적절한 레벨 활용

### 일반 RAG vs RAPTOR

| 특징 | 일반 RAG | RAPTOR |
|------|---------|--------|
| 검색 대상 | 원본 청크만 | 원본 + 여러 레벨 요약 |
| 개괄적 질문 | 어려움 | 상위 요약에서 답변 |
| 세부적 질문 | 가능 | 원본 청크에서 답변 |
| 긴 문서 처리 | 제한적 | 효과적 |

### 실전 활용

- **연구 논문 분석**: 전체 내용 파악 + 세부 실험 질의
- **긴 보고서**: 요약 + 상세 데이터 검색
- **법률 문서**: 전체 맥락 + 특정 조항 검색

### 확장 가능성

이 노트북은 2레벨 구조를 구현했지만, 실제 RAPTOR는:
- 3~4개 레벨 사용 가능
- 클러스터링으로 관련 청크 그룹화
- 트리 구조로 계층 관리

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 원본 청크(레벨 0) + 그룹 요약(레벨 1)을 하나의 벡터스토어에 통합 |
| 검색 장점 | 세부 내용 질문엔 레벨 0, 전체 개요 질문엔 레벨 1 문서가 검색됨 |
| 이 노트북의 범위 | 간소화 버전 — 클러스터링·다단계 요약은 원논문 참조 |
| 구현 핵심 | `sum_docs` 리스트에 원본+요약 모두 추가 후 `FAISS.from_documents()` |
| 주의 | 요약 품질이 LLM 성능에 크게 의존 — 요약 프롬프트 튜닝이 중요 |

### RAPTOR 레벨 구조

| 레벨 | 내용 | 적합한 질문 유형 |
|------|------|-----------------|
| 레벨 0 | 원본 청크 (세부 정보) | "3장 2절의 구체적 내용은?" |
| 레벨 1 | 그룹 요약 (핵심 정리) | "이 문서의 전체 주제는?" |
| 레벨 2+ | 상위 요약 (원논문) | 더 넓은 추상화 — 이 노트북 범위 외 |

### 다음 노트북 예고

**05-Web-Summarization** — Map-Reduce 패턴으로 웹 문서를 대규모로 요약하고, 원본과 요약 모두를 벡터스토어에 저장해 질문 유형에 따라 최적 문서를 반환하는 방법을 배워요.